# Nikkei 225 Buzz & Price Tracker

This notebook traces the stock price and news "buzz" (volume) for major Nikkei 225 companies. It combines data collection and visualization into a single workflow.

In [ ]:
# Install dependencies if not already installed
%pip install yfinance feedparser pandas plotly requests beautifulsoup4

In [ ]:
import yfinance as yf
import feedparser
import pandas as pd
import datetime
import os
import urllib.parse
import time
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

In [ ]:
# --- Configuration ---

# Nikkei 225 subset for demonstration
TICKERS = {
    "7203.T": "トヨタ自動車",
    "6758.T": "ソニーグループ",
    "9984.T": "ソフトバンクグループ",
    "9983.T": "ファーストリテイリング",
    "8035.T": "東京エレクトロン",
    "6861.T": "キーエンス",
    "7974.T": "任天堂",
    "9432.T": "日本電信電話",
    "6501.T": "日立製作所",
    "8306.T": "三菱UFJフィナンシャル・グループ"
}

DATA_FILE = "market_data.csv"

In [ ]:
# --- Data Collection Functions ---

def fetch_stock_price(ticker):
    """
    Fetches the current stock price.
    """
    try:
        stock = yf.Ticker(ticker)
        # Try fast_info first
        try:
            price = stock.fast_info.last_price
        except:
            # Fallback to history
            hist = stock.history(period="1d")
            if not hist.empty:
                price = hist['Close'].iloc[-1]
            else:
                return None
        return price
    except Exception as e:
        print(f"Error fetching stock for {ticker}: {e}")
        return None

def fetch_buzz_and_sentiment(company_name):
    """
    Fetches news from Google News RSS for the company name.
    Returns:
        buzz_score (int): Number of articles found (proxy for attention).
        sentiment_score (float): Simple keyword-based sentiment.
        top_headline (str): The most recent headline.
    """
    encoded_query = urllib.parse.quote(company_name)
    rss_url = f"https://news.google.com/rss/search?q={encoded_query}&hl=ja&gl=JP&ceid=JP:ja"
    
    try:
        feed = feedparser.parse(rss_url)
        entries = feed.entries
        
        buzz_score = len(entries)
        
        # Simple sentiment heuristic for Japanese titles
        positive_keywords = ["急騰", "上昇", "好調", "最高益", "増益", "買収", "提携", "ストップ高", "成長", "期待"]
        negative_keywords = ["急落", "下落", "不調", "赤字", "減益", "中止", "撤退", "ストップ安", "懸念", "失望"]
        
        total_score = 0
        for entry in entries:
            title = entry.title
            score = 0
            for pk in positive_keywords:
                if pk in title:
                    score += 1
            for nk in negative_keywords:
                if nk in title:
                    score -= 1
            total_score += score
            
        avg_sentiment = total_score / buzz_score if buzz_score > 0 else 0
        
        if entries:
            top_headline = entries[0].title
        else:
            top_headline = "No recent news"
        
        return buzz_score, avg_sentiment, top_headline
        
    except Exception as e:
        print(f"Error fetching news for {company_name}: {e}")
        return 0, 0, "Error"

In [ ]:
# --- Execution: Collect Data ---

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
new_rows = []

print(f"Starting data update at {timestamp}...")

# Load existing data if available
existing_data = []
if os.path.exists(DATA_FILE):
    try:
        df_existing = pd.read_csv(DATA_FILE)
        existing_data = df_existing.to_dict('records')
    except Exception as e:
        print(f"Error reading existing data: {e}")

for ticker, name in TICKERS.items():
    print(f"Processing {name} ({ticker})...")
    
    price = fetch_stock_price(ticker)
    if price is None:
         print(f"Skipping {name} due to stock fetch error.")
         continue

    buzz, sentiment, headline = fetch_buzz_and_sentiment(name)
    
    new_rows.append({
        "timestamp": timestamp,
        "ticker": ticker,
        "name": name,
        "price": price,
        "buzz_score": buzz,
        "sentiment_score": sentiment,
        "top_headline": headline
    })
    time.sleep(1) # Be nice to APIs
        
if new_rows:
    df_new = pd.DataFrame(new_rows)
    if existing_data:
        df_combined = pd.concat([pd.DataFrame(existing_data), df_new], ignore_index=True)
    else:
        df_combined = df_new
        
    # Ensure numeric columns are correct type
    df_combined['price'] = pd.to_numeric(df_combined['price'], errors='coerce')
    df_combined['buzz_score'] = pd.to_numeric(df_combined['buzz_score'], errors='coerce')
    df_combined['sentiment_score'] = pd.to_numeric(df_combined['sentiment_score'], errors='coerce')

    df_combined.to_csv(DATA_FILE, index=False)
    print("Data update complete and saved to CSV.")
else:
    print("No new data collected.")
    if existing_data:
        df_combined = pd.DataFrame(existing_data)
    else:
        df_combined = pd.DataFrame()

In [ ]:
# --- Visualization ---

if not df_combined.empty:
    df_combined['timestamp'] = pd.to_datetime(df_combined['timestamp'])
    
    # 1. Stock Price History
    fig_price = px.line(
        df_combined, 
        x="timestamp", 
        y="price", 
        color="name", 
        markers=True,
        title="Stock Price History (JPY)"
    )
    fig_price.show()
    
    # 2. Buzz Volume History
    fig_buzz = px.bar(
        df_combined, 
        x="timestamp", 
        y="buzz_score", 
        color="name", 
        barmode="group",
        title="News Buzz Volume History"
    )
    fig_buzz.show()
    
    # 3. Sentiment History
    fig_sent = px.line(
        df_combined, 
        x="timestamp", 
        y="sentiment_score", 
        color="name", 
        markers=True,
        title="Sentiment Score History"
    )
    fig_sent.show()
    
    # 4. Latest Headlines Table
    display(Markdown("### Latest Headlines"))
    latest_news = df_combined.sort_values("timestamp", ascending=False).drop_duplicates("ticker")
    display(latest_news[['name', 'top_headline', 'timestamp']])
else:
    print("No data available to visualize.")

# Correlation Analysis (Last 30 Days)

This section analyzes the relationship between daily stock prices and news buzz volume for the past 30 days.

**Note:** This analysis relies on accumulated historical data in `market_data.csv`. If you are running this for the first time, correlations will not be available or meaningful until sufficient daily data points are collected.

In [ ]:
if not df_combined.empty:
    # 1. Filter for Last 30 Days
    thirty_days_ago = pd.Timestamp.now() - pd.Timedelta(days=30)
    df_30d = df_combined[df_combined['timestamp'] >= thirty_days_ago].copy()
    
    # 2. Resample to Daily Frequency
    # We group by Ticker and Day. 
    # Price: Take the LAST observed price of the day.
    # Buzz: Take the MEAN (average) buzz score observed that day.
    df_30d['date'] = df_30d['timestamp'].dt.date
    daily_stats = df_30d.groupby(['ticker', 'name', 'date']).agg({
        'price': 'last',
        'buzz_score': 'mean'
    }).reset_index()
    
    # 3. Calculate Correlation per Ticker
    correlations = []
    for ticker in daily_stats['ticker'].unique():
        subset = daily_stats[daily_stats['ticker'] == ticker]
        name = subset['name'].iloc[0]
        
        # Need at least 2 points for correlation
        if len(subset) > 1:
            corr = subset['price'].corr(subset['buzz_score'])
            correlations.append({"ticker": ticker, "name": name, "correlation": corr, "count": len(subset)})
        else:
            correlations.append({"ticker": ticker, "name": name, "correlation": 0, "count": len(subset)})
            
    df_corr = pd.DataFrame(correlations).sort_values("correlation", ascending=False)
    
    display(Markdown("### Price vs. Buzz Correlation (Daily Aggregated)"))
    display(df_corr)
    
    # 4. Visualization: Correlation Bar Chart
    if not df_corr.empty:
        fig_corr = px.bar(
            df_corr, 
            x="name", 
            y="correlation", 
            title="Correlation: Daily Stock Price vs. Buzz Volume (Last 30 Days)",
            color="correlation",
            color_continuous_scale="RdBu",
            range_color=[-1, 1]
        )
        fig_corr.show()
    
    # 5. Visualization: Scatter Plot (All Data Points)
    # Showing the raw relationship
    if not daily_stats.empty:
        fig_scatter = px.scatter(
            daily_stats, 
            x="buzz_score", 
            y="price", 
            color="name", 
            title="Scatter Plot: Daily Buzz vs. Price",
            hover_data=['date']
        )
        fig_scatter.show()
else:
    print("No data available for correlation analysis.")